# Day 5 · 机器学习流程与数据划分

目标：定义预测问题，区分特征和标签，解释训练/验证/测试的职责。
基础 45 分钟、深度 40 分钟、广度 25 分钟、作业 30 分钟；Bonus 30 分钟。

### 1. 任务类型

| 任务 | 是否有标签 | 输出 | 本周例子 |
|---|---|---|---|
| 监督回归 | 有 | 连续数值 | 预测 G3 |
| 监督分类 | 有 | 类别 | 预测及格与否 |
| 无监督聚类 | 无 | 相似样本的分组 | 按学习行为分群 |

分类的准确率不能直接用于评价回归；回归用误差指标。
一行是样本，一列是特征。G3 是目标，不能出现在 X 中。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course/bootcamp/week1/data/raw/student-mat.csv").exists())
raw = pd.read_csv(ROOT / "course/bootcamp/week1/data/raw/student-mat.csv", sep=";")
# 固定划分，第三至七天始终使用相同学生；实际行数为 276/59/60。
train, heldout = train_test_split(raw, test_size=0.30, random_state=42)
val, test = train_test_split(heldout, test_size=0.50, random_state=42)
train, val, test = train.copy(), val.copy(), test.copy()
print("训练/验证/测试：", len(train), len(val), len(test))


In [ ]:
features = ["G1","G2","studytime","failures"]
X_train, y_train = train[features], train["G3"]
print("X:", X_train.shape, "y:", y_train.shape)
assert "G3" not in X_train.columns
display(X_train.head())
display(y_train.head())

### 2. 拆分代码逐行理解

第一次 test_size=0.30 保留约 30% 作为临时集。
第二次将临时集平分成验证和测试。由于人数必须为整数，结果不是精确的 70/15/15。
random_state 固定随机序列，保证每次实验可比较；它不是一个要调到分数更高的参数。

训练集：拟合参数和统计量。验证集：选择方案。测试集：方案冻结后的最终估计。
第三至四天已预先采用相同划分，因此它们的探索没有看见测试标签。

In [ ]:
sets = [set(part.index) for part in [train,val,test]]
assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])
assert set.union(*sets) == set(raw.index)
display(pd.DataFrame({"rows":[len(train),len(val),len(test)],
                      "fraction":[len(train)/len(raw),len(val)/len(raw),len(test)/len(raw)]},
                     index=["train","validation","test"]))

### 3. 深度：预测时点决定合法特征

本周主线：假设第二阶段成绩已知，估计期末 G3，可以使用 G1/G2。
开学时预测：G1/G2 尚未产生，必须去掉。
期末累计缺勤数 absences 也可能包含预测之后的信息，主线不使用它。
数据没有完整时间戳，因此这里验证的是离线教学方案，不能据此宣称已具备真实预警效果。

同一个学生跨课程出现时，不能简单拼接两份数据再随机拆分；同一人的记录应分在同一组。
未来学期预测应考虑按时间划分，新学校泛化应考虑按学校留出。

### 4. 数据泄漏演示

以下仅用人工数组演示错误：如果用训练和未来数据一起算均值，未来分布便影响了变换。
正式模型会用 Pipeline，fit 只接收训练集。

In [ ]:
training_values = np.array([1.,2.,3.])
future_values = np.array([100.,101.])
print("仅训练均值：", training_values.mean())
print("错误的混合均值：", np.concatenate([training_values,future_values]).mean())
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
imputer.fit(train[features])
display(pd.Series(imputer.statistics_, index=features))
print("验证特征转换形状：", imputer.transform(val[features]).shape)

### 5. 广度：先建立基准模型

DummyRegressor(strategy="mean") 始终预测训练目标均值。
这是衡量复杂模型是否有价值的起点。均值是平方损失下的最优常数；MAE 对应的最优常数是中位数。

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
for strategy in ["mean", "median"]:
    base = DummyRegressor(strategy=strategy).fit(train[features], train["G3"])
    pred = base.predict(val[features])
    print(strategy, "validation MAE:", mean_absolute_error(val["G3"], pred))

### DAY 5 ASSIGNMENT

基础：用自己的话解释三份数据的职责，计算实际比例。

深度：判断以下是否泄漏并说明原因：用全数据填缺失；把 G3 放进 X；开学时用 G2；用验证集选择模型。

广度：如果部署到全新学校，随机拆分是否足够？

Bonus：在训练集上建立 5 折 KFold，检查每折不重叠；此时不要动最终测试集。

验收：特征不含标签、三份行索引互斥、随机种子固定、写清预测时点。

In [ ]:
# 作业区


### 参考答案

前面三个做法存在泄漏；用验证集选择模型是正常用途，但反复试很多方案也会过拟合验证集。
KFold 仅在训练集内部产生多次训练/验证组合，最终测试仍封存。

In [ ]:
from sklearn.model_selection import KFold
folds = KFold(n_splits=5, shuffle=True, random_state=42)
for i, (fit_idx, check_idx) in enumerate(folds.split(X_train), 1):
    assert not set(fit_idx) & set(check_idx)
    print("fold", i, "fit", len(fit_idx), "check", len(check_idx))

### 复盘（填写）

我的预测对象、预测时点、特征和标签：___。最容易发生泄漏的一步：___。